## Libraries

In [3]:
# SQLite driver used as the main analytical storage backend
import sqlite3

# Path handling for filesystem-safe and cross-platform paths
from pathlib import Path

# Core data manipulation library
import pandas as pd

# Date and time utilities for timestamps and temporal analysis
from datetime import datetime

# Operating system utilities (paths, environment variables, filesystem ops)
import os

# Mathematical utilities for aggregation and layout calculations
import math

# Core data manipulation library (used throughout the EDA pipeline)
import pandas as pd

# Numerical computing library for vectorized operations
import numpy as np

# Plotting library for static visualizations
import matplotlib.pyplot as plt

# High-level visualization library built on top of matplotlib
import seaborn as sns

# Statistical functions used for hypothesis testing
from scipy import stats

# Warning management to suppress non-critical runtime messages
import warnings


# Normalize a keyword into a compact identifier.
# This helper lowercases the input and removes all
# spaces and hyphens to produce a single-token string.
# Input : keyword (str)
# Output: str   compact normalized keyword
def normalize_keyword(keyword: str) -> str:
    return (keyword.lower().strip().replace(" ", "").replace("-", ""))

## Youtube Data Quality checks and Integration

In [ ]:
def clean_youtube_data(topic: str, storage_path: str | Path, output_dir: str | Path):

    DB_PATH = Path(storage_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    prefix = normalize_keyword(topic)

    YT_VIDEOS             = f"{prefix}_youtube_videos"
    YT_VIDEO_STATS        = f"{prefix}_youtube_video_stats"
    YT_CHANNEL_STATS      = f"{prefix}_youtube_channel_stats"

    YT_VIDEOS_INCOMPLETE  = f"{prefix}_youtube_videos_incomplete"
    YT_CHANNEL_INCOMPLETE = f"{prefix}_youtube_channel_incomplete"
    YT_FULL_STATS         = f"{prefix}_youtube_full_statistics"

    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        cursor = conn.cursor()

        # Identify videos with missing or inconsistent statistics
        cursor.execute(f"DROP TABLE IF EXISTS {YT_VIDEOS_INCOMPLETE}")
        cursor.execute(f"""
            CREATE TABLE {YT_VIDEOS_INCOMPLETE} AS
            SELECT DISTINCT video_id
            FROM {YT_VIDEO_STATS}
            WHERE
                view_count IS NULL
                OR like_count IS NULL
                OR comment_count IS NULL
                OR duration_seconds IS NULL
                OR like_count > view_count
                OR view_count < 0
                OR like_count < 0
                OR comment_count < 0
        """)

        cursor.execute(f"SELECT COUNT(*) FROM {YT_VIDEOS_INCOMPLETE}")
        n_vid_inc = cursor.fetchone()[0]
        
        cursor.execute(f"SELECT COUNT(*) FROM {YT_VIDEO_STATS}")
        n_vid_tot = cursor.fetchone()[0]
        
        pct_vid_inc = (n_vid_inc / n_vid_tot * 100) if n_vid_tot > 0 else 0.0
        
        print(f"Incomplete videos: {n_vid_inc} ({pct_vid_inc:.2f}% of total videos)")


        # Identify channels with missing or hidden subscriber information
        cursor.execute(f"DROP TABLE IF EXISTS {YT_CHANNEL_INCOMPLETE}")
        cursor.execute(f"""
            CREATE TABLE {YT_CHANNEL_INCOMPLETE} AS
            SELECT DISTINCT channel_id
            FROM {YT_CHANNEL_STATS}
            WHERE subscriber_count IS NULL OR hidden_subscribers > 0
        """)
        
        cursor.execute(f"SELECT COUNT(*) FROM {YT_CHANNEL_INCOMPLETE}")
        n_ch_inc = cursor.fetchone()[0]
        
        cursor.execute(f"SELECT COUNT(*) FROM {YT_CHANNEL_STATS}")
        n_ch_tot = cursor.fetchone()[0]
        
        pct_ch_inc = (n_ch_inc / n_ch_tot * 100) if n_ch_tot > 0 else 0.0
        
        print(f"Incomplete channels: {n_ch_inc} ({pct_ch_inc:.2f}% of total channels)")


        # Build the clean joined YouTube dataset
        cursor.execute(f"DROP TABLE IF EXISTS {YT_FULL_STATS}")
        cursor.execute(f"""
            CREATE TABLE {YT_FULL_STATS} AS
            SELECT
                v.video_id,
                v.channel_id,
                v.published_at,
                vs.view_count           AS video_view_count,
                vs.like_count,
                vs.comment_count,
                vs.duration_seconds,
                vs.is_reel,
                vs.collected_at         AS video_collected_at,
                cs.subscriber_count,
                cs.video_count          AS channel_video_count,
                cs.view_count           AS channel_view_count
            FROM {YT_VIDEOS} v
            LEFT JOIN {YT_VIDEO_STATS} vs
                ON v.video_id = vs.video_id
            LEFT JOIN {YT_CHANNEL_STATS} cs
                ON v.channel_id = cs.channel_id
            WHERE
                v.video_id NOT IN (SELECT video_id FROM {YT_VIDEOS_INCOMPLETE})
                AND v.channel_id NOT IN (SELECT channel_id FROM {YT_CHANNEL_INCOMPLETE})
                AND vs.view_count > 50
        """)
        cursor.execute(f"SELECT COUNT(*) FROM {YT_FULL_STATS}")
        n_final = cursor.fetchone()[0]
        
        print(f"Final clean videos: {n_final}")
        print("")


        conn.commit()


## Youtube Data Enrichment

In [4]:
def analyze_youtube_data(topic: str, storage_path: str | Path, output_dir: str | Path):

    DB_PATH = Path(storage_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    prefix = normalize_keyword(topic)

    YT_FULL_STATS         = f"{prefix}_youtube_full_statistics"
    INTERESTING_VIDEO_ID  = f"{prefix}_interesting_videos_id"
    INTERESTING_STATS     = f"{prefix}_interesting_statistics"
    INTERESTING_VIDEO     = f"{prefix}_interesting_video"
    INTERESTING_REEL      = f"{prefix}_interesting_reel"
    BEST_SCORE_VIDEO      = f"{prefix}_best_score_videos"
    TOP_OPPORTUNITY_VIDEO = f"{prefix}_top_opportunity_videos"

    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        cursor = conn.cursor()

        # Exposure efficiency: views relative to channel size
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN exposure_efficiency REAL")
        cursor.execute(f"""
            UPDATE {YT_FULL_STATS}
            SET exposure_efficiency =CASE
                                        WHEN subscriber_count > 0
                                        THEN CAST(video_view_count AS REAL) / subscriber_count
                                        ELSE NULL
                                     END
        """)

        # Engagement rate: likes + comments per view
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN engagement_rate REAL")
        cursor.execute(f"""
            UPDATE {YT_FULL_STATS}
            SET engagement_rate =CASE
                                    WHEN video_view_count > 0
                                    THEN CAST(like_count + comment_count AS REAL) / video_view_count
                                    ELSE NULL
                                 END
        """)

        # View percentile
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN view_percentile REAL")
        cursor.execute(f"""
        WITH ranked_views AS (
            SELECT video_id, PERCENT_RANK() OVER (ORDER BY video_view_count) AS pct
            FROM {YT_FULL_STATS})
        UPDATE {YT_FULL_STATS}
        SET view_percentile = (SELECT pct
                               FROM ranked_views
                               WHERE ranked_views.video_id = {YT_FULL_STATS}.video_id)
        """)


        # Engagement percentile
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN engagement_rate_pct REAL")
        cursor.execute(f"""
        WITH ranked_engagement AS (SELECT video_id, PERCENT_RANK() OVER (ORDER BY engagement_rate) AS pct
                                   FROM {YT_FULL_STATS})
        UPDATE {YT_FULL_STATS}
        SET engagement_rate_pct = (SELECT pct
                                   FROM ranked_engagement
                                   WHERE ranked_engagement.video_id = {YT_FULL_STATS}.video_id)
        """)


        # Exposure percentile
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN exposure_efficiency_pct REAL")
        cursor.execute(f"""
        WITH ranked_exposure AS (SELECT video_id, PERCENT_RANK() OVER (ORDER BY exposure_efficiency) AS pct
                                 FROM {YT_FULL_STATS})
        UPDATE {YT_FULL_STATS}
        SET exposure_efficiency_pct = (SELECT pct
                                       FROM ranked_exposure
                                       WHERE ranked_exposure.video_id = {YT_FULL_STATS}.video_id)
        """)


        # Best content score
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN best_content_score REAL")
        cursor.execute(f"""
            UPDATE {YT_FULL_STATS}
            SET best_content_score = 0.40 * engagement_rate_pct +
                                     0.30 * exposure_efficiency_pct +
                                     0.30 * view_percentile
        """)

        # Opportunity score
        cursor.execute(f"ALTER TABLE {YT_FULL_STATS} ADD COLUMN opportunity_score REAL")
        cursor.execute(f"""
            UPDATE {YT_FULL_STATS}
            SET opportunity_score = 0.40 * engagement_rate_pct +
                                    0.40 * (1.0 - exposure_efficiency_pct) +
                                    0.20 * (1.0 - view_percentile)
        """)

        # Top 20% by best score
        cursor.execute(f"DROP TABLE IF EXISTS {BEST_SCORE_VIDEO}")
        cursor.execute(f"""
            CREATE TABLE {BEST_SCORE_VIDEO} AS
            SELECT video_id, is_reel
            FROM {YT_FULL_STATS}
            ORDER BY best_content_score DESC
            LIMIT CAST(0.2 * (SELECT COUNT(*) FROM {YT_FULL_STATS}) AS INTEGER)
        """)

        # Top 20% by opportunity
        cursor.execute(f"DROP TABLE IF EXISTS {TOP_OPPORTUNITY_VIDEO}")
        cursor.execute(f"""
            CREATE TABLE {TOP_OPPORTUNITY_VIDEO} AS
            SELECT video_id, is_reel
            FROM {YT_FULL_STATS}
            ORDER BY opportunity_score DESC
            LIMIT CAST(0.2 * (SELECT COUNT(*) FROM {YT_FULL_STATS}) AS INTEGER)
        """)

        # Union of both selections
        cursor.execute(f"DROP TABLE IF EXISTS {INTERESTING_VIDEO_ID}")
        cursor.execute(f"""
            CREATE TABLE {INTERESTING_VIDEO_ID} AS
            SELECT video_id,is_reel,MAX(from_best) AS from_best,MAX(from_opportunity) AS from_opportunity,
                   MAX(from_best) + MAX(from_opportunity) AS source_score
            FROM (SELECT video_id, is_reel,1 AS from_best,0 AS from_opportunity
                FROM {BEST_SCORE_VIDEO}
                UNION ALL
                SELECT video_id, is_reel,0 AS from_best,1 AS from_opportunity
                FROM {TOP_OPPORTUNITY_VIDEO})
            GROUP BY video_id, is_reel

        """)

        cursor.execute(f"SELECT COUNT(*) FROM {INTERESTING_VIDEO_ID}")
        n_interesting = cursor.fetchone()[0]
        
        print(f"Total interesting videos (union): {n_interesting}")
        print("")


        # Extract full stats
        cursor.execute(f"DROP TABLE IF EXISTS {INTERESTING_STATS}")
        cursor.execute(f"""
            CREATE TABLE {INTERESTING_STATS} AS
            SELECT f.*
            FROM {YT_FULL_STATS} f
            JOIN {INTERESTING_VIDEO_ID} i
            ON f.video_id = i.video_id
        """)

        # Separate reels and videos
        cursor.execute(f"DROP TABLE IF EXISTS {INTERESTING_REEL}")
        cursor.execute(f"""
            CREATE TABLE {INTERESTING_REEL} AS
            SELECT v.title, v.description, i.source_score
            FROM {prefix}_youtube_videos v
            JOIN {INTERESTING_VIDEO_ID} i
            ON v.video_id = i.video_id
            WHERE i.is_reel = 1
        """)

        cursor.execute(f"DROP TABLE IF EXISTS {INTERESTING_VIDEO}")
        cursor.execute(f"""
            CREATE TABLE {INTERESTING_VIDEO} AS
            SELECT v.title, v.description, i.source_score
            FROM {prefix}_youtube_videos v
            JOIN {INTERESTING_VIDEO_ID} i
            ON v.video_id = i.video_id
            WHERE i.is_reel = 0
        """)

        conn.commit()


## Google Rising Queries Cleaning

In [ ]:
def clean_google_trends_rising(topic: str,storage_path: str | Path,output_dir: str | Path):

    DB_PATH = Path(storage_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    prefix = normalize_keyword(topic)

    GOOGLE_RISING = f"{prefix}_google_trends_rising"
    GOOGLE_RISING_CLEANED = f"{prefix}_google_rising_cleaned"

    # Flag raised if the dataset looks incomplete
    incomplete_flag = 0

    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        cursor = conn.cursor()

        print("\nCleaning Google Trends RISING queries")

        # Drop to guarantee a clean rebuild
        cursor.execute(f"DROP TABLE IF EXISTS {GOOGLE_RISING_CLEANED}")

        # Normalize increase_percent and explicitly flag breakout events
        cursor.execute(f"""
            CREATE TABLE {GOOGLE_RISING_CLEANED} AS
            SELECT
                query,
                window_days AS extraction_window,
                search_interest,

                -- Numeric version of growth signal
                CASE
                    WHEN LOWER(TRIM(increase_percent)) = 'breakout' THEN 5000.0
                    WHEN TRIM(increase_percent) GLOB '-[0-9]*.[0-9]*%'
                      OR TRIM(increase_percent) GLOB '[0-9]*.[0-9]*%'
                      OR TRIM(increase_percent) GLOB '-[0-9]*%'
                      OR TRIM(increase_percent) GLOB '[0-9]*%'
                    THEN CAST(REPLACE(TRIM(increase_percent), '%', '') AS REAL)
                    ELSE NULL
                END AS increase_pct,

                -- Binary breakout flag used in later analysis
                CASE
                    WHEN LOWER(TRIM(increase_percent)) = 'breakout' THEN 1
                    WHEN (
                        TRIM(increase_percent) GLOB '-[0-9]*.[0-9]*%'
                        OR TRIM(increase_percent) GLOB '[0-9]*.[0-9]*%'
                        OR TRIM(increase_percent) GLOB '-[0-9]*%'
                        OR TRIM(increase_percent) GLOB '[0-9]*%'
                    )
                    AND CAST(REPLACE(TRIM(increase_percent), '%', '') AS REAL) >= 150
                    THEN 1
                    ELSE 0
                END AS breakout

            FROM {GOOGLE_RISING}
            WHERE window_days IN (60, 30, 14, 7)
        """)

        cursor.execute(f"SELECT COUNT(*) FROM {GOOGLE_RISING_CLEANED}")
        n_rows = cursor.fetchone()[0]
        print(f"Google Rising cleaned rows: {n_rows}")

        # Check which extraction windows are present (RISING)
        expected_windows = {7, 14, 30, 60}
        
        cursor.execute(f"""SELECT DISTINCT window_days FROM {GOOGLE_RISING}""")
        found_windows = {row[0] for row in cursor.fetchall()}
        
        missing_windows = expected_windows - found_windows
        
        if missing_windows:
            incomplete_flag = 1
            print("\nATTENTION: Google Trends RISING dataset looks incomplete.")
            print(f"Missing extraction windows: {sorted(missing_windows)}\n")

        conn.commit()
        
    return incomplete_flag

## Google Top Queries Cleaning

In [ ]:
def clean_google_trends_top(topic: str,storage_path: str | Path,output_dir: str | Path):

    DB_PATH = Path(storage_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    prefix = normalize_keyword(topic)

    GOOGLE_TOP = f"{prefix}_google_trends_top"
    GOOGLE_TOP_CLEANED = f"{prefix}_google_top_cleaned"

    # Flag raised if the dataset looks incomplete
    incomplete_flag = 0

    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        cursor = conn.cursor()

        print("\nCleaning Google Trends TOP queries")

        cursor.execute(f"DROP TABLE IF EXISTS {GOOGLE_TOP_CLEANED}")

        # Normalize increase_percent and keep only short-term windows
        cursor.execute(f"""
            CREATE TABLE {GOOGLE_TOP_CLEANED} AS
            SELECT
                query,
                window_days AS extraction_window,
                search_interest,

                CASE
                    WHEN LOWER(TRIM(increase_percent)) = 'breakout' THEN 999.0
                    WHEN TRIM(increase_percent) GLOB '-[0-9]*.[0-9]*%'
                      OR TRIM(increase_percent) GLOB '[0-9]*.[0-9]*%'
                      OR TRIM(increase_percent) GLOB '-[0-9]*%'
                      OR TRIM(increase_percent) GLOB '[0-9]*%'
                    THEN CAST(REPLACE(TRIM(increase_percent), '%', '') AS REAL)
                    ELSE NULL
                END AS increase_pct

            FROM {GOOGLE_TOP}
            WHERE window_days IN (7, 14)
        """)

        cursor.execute(f"SELECT COUNT(*) FROM {GOOGLE_TOP_CLEANED}")
        n_rows = cursor.fetchone()[0]
        print(f"Google Top cleaned rows: {n_rows}")

        # Checking which extraction windows are present
        expected_windows = {7, 14}
        
        cursor.execute(f"""SELECT DISTINCT extraction_window FROM {GOOGLE_TOP_CLEANED}""")
        found_windows = {row[0] for row in cursor.fetchall()}
        
        missing_windows = expected_windows - found_windows
        
        if missing_windows:
            incomplete_flag = 1
            print("\nATTENTION: Google Trends TOP dataset looks incomplete.")
            print(f"Missing extraction windows: {sorted(missing_windows)}\n")

        conn.commit()

    return incomplete_flag

## Google Rising Queries Analysis

In [ ]:
def analyze_google_trends_rising(topic: str,storage_path: str | Path,output_dir: str | Path):

    DB_PATH = Path(storage_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    prefix = normalize_keyword(topic)

    GOOGLE_RISING_CLEANED = f"{prefix}_google_rising_cleaned"
    GOOGLE_RISING_COMPUTED= f"{prefix}_google_rising_computed"
    GOOGLE_RISING_RANKED  = f"{prefix}_google_rising_ranked"

    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        cursor = conn.cursor()

        print("\nAnalyzing Google Trends RISING patterns")

        cursor.execute(f"DROP TABLE IF EXISTS {GOOGLE_RISING_COMPUTED}")

        # Build temporal fingerprints across extraction windows
        cursor.execute(f"""
            CREATE TABLE {GOOGLE_RISING_COMPUTED} AS
            WITH breakout_rows AS (
                SELECT
                    query,
                    extraction_window,
                    CASE
                        WHEN extraction_window = 7  THEN 8
                        WHEN extraction_window = 14 THEN 4
                        WHEN extraction_window = 30 THEN 2
                        WHEN extraction_window = 60 THEN 1
                        ELSE 0
                    END * LOG(1 + increase_pct) AS row_score
                FROM {GOOGLE_RISING_CLEANED}
                WHERE breakout = 1),
            pivoted AS (
                SELECT
                    query,
                    MAX(CASE WHEN extraction_window = 60 THEN 1 ELSE 0 END) AS b60,
                    MAX(CASE WHEN extraction_window = 30 THEN 1 ELSE 0 END) AS b30,
                    MAX(CASE WHEN extraction_window = 14 THEN 1 ELSE 0 END) AS b14,
                    MAX(CASE WHEN extraction_window = 7  THEN 1 ELSE 0 END) AS b7,
                    SUM(row_score) AS row_score
                FROM breakout_rows
                GROUP BY query)
            SELECT
                query,
                b60, b30, b14, b7,
                CAST(b60 AS TEXT) || CAST(b30 AS TEXT) || CAST(b14 AS TEXT) || CAST(b7 AS TEXT) AS pattern,
                row_score
            FROM pivoted
        """)

        print("Ranking Google Trends RISING queries")

        cursor.execute(f"DROP TABLE IF EXISTS {GOOGLE_RISING_RANKED}")

        cursor.execute(f"""
            CREATE TABLE {GOOGLE_RISING_RANKED} AS
            WITH ranked AS (
                SELECT *,
                    CASE pattern
                    -- Only present in the last 7 days: brand-new spike
                    WHEN '0001' THEN 1
                    -- Present in 60d and 7d: old topic resurfacing strongly
                    WHEN '1001' THEN 2
                    -- Present in 30d and 7d: medium-term trend with recent acceleration
                    WHEN '0101' THEN 3
                    -- Present in 60d, 30d and 7d, missing 14d
                    WHEN '1101' THEN 4
                    -- Present in 60d, 14d and 7d, missing 30d
                    WHEN '1011' THEN 5
                    -- Present only in the last two windows (14d and 7d)
                    WHEN '0011' THEN 6
                    -- Present in 30d, 14d and 7d: sustained recent growth
                    WHEN '0111' THEN 7
                    -- Present in all windows: long-term sustained growth
                    WHEN '1111' THEN 8
                    -- Present in all but the most recent window: fading trend
                    WHEN '1110' THEN 9
                    -- Intermittent spikes (60d and 14d)
                    WHEN '1010' THEN 10
                    -- Present only in older windows (60d and 30d)
                    WHEN '1100' THEN 11
                    -- Present only in the 14-day window
                    WHEN '0010' THEN 12
                    -- Present in 30d and 14d, missing recent spike
                    WHEN '0110' THEN 13
                    -- Present only in the 30-day window
                    WHEN '0100' THEN 14
                    -- Present only in the 60-day window: very old / decaying trend
                    WHEN '1000' THEN 15
                    -- No breakout detected (should not happen after filtering)
                    WHEN '0000' THEN 16
                    -- Fallback for unexpected patterns
                    ELSE 99
                    END AS pattern_rank
                FROM {GOOGLE_RISING_COMPUTED}),
            ordered AS (
                SELECT *,
                       ROW_NUMBER() OVER (ORDER BY pattern_rank, row_score DESC) AS rn,
                       COUNT(*) OVER () AS total_rows
                FROM ranked)
            SELECT query
            FROM ordered
            WHERE rn <= CASE
                WHEN total_rows < 5 THEN 1
                ELSE CAST(0.3 * total_rows AS INTEGER)
            END
            ORDER BY rn
        """)

        conn.commit()

## Google Top Queries Analysis

In [ ]:
def analyze_google_trends_top(topic: str,storage_path: str | Path,output_dir: str | Path):

    # Resolve database path and ensure output directory exists
    DB_PATH = Path(storage_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Normalize topic to ensure consistency with SQL table naming
    prefix = normalize_keyword(topic)

    # Define all table names used in this analysis
    GOOGLE_TOP_CLEANED  = f"{prefix}_google_top_cleaned"
    GOOGLE_TOP_COMPUTED = f"{prefix}_google_top_computed"
    GOOGLE_TOP_RANKED   = f"{prefix}_google_top_ranked"

    # Open a SQLite connection with a safe timeout for heavy queries
    with sqlite3.connect(DB_PATH, timeout=30) as conn:
        cursor = conn.cursor()

        print("\nAnalyzing Google Trends TOP queries")

        # Multiple rows generated by different extraction windows are aggregated into a single row per query.
        # Presence in the 7-day and 14-day windows and the corresponding metrics are explicitly tracked to
        # construct a temporal fingerprint for each query.
        cursor.execute(f"DROP TABLE IF EXISTS {GOOGLE_TOP_COMPUTED}")

        cursor.execute(f"""
            CREATE TABLE {GOOGLE_TOP_COMPUTED} AS
            WITH pivoted AS (
                SELECT
                    query,

                    -- Presence flags: used later to build temporal patterns
                    MAX(CASE WHEN extraction_window = 7  THEN 1 ELSE 0 END)  AS b7,
                    MAX(CASE WHEN extraction_window = 14 THEN 1 ELSE 0 END) AS b14,

                    -- Metrics observed in the 7-day window
                    MAX(CASE WHEN extraction_window = 7
                             THEN search_interest END) AS search_interest7,
                    MAX(CASE WHEN extraction_window = 7
                             THEN increase_pct END)    AS increase_pct7,

                    -- Metrics observed in the 14-day window
                    MAX(CASE WHEN extraction_window = 14
                             THEN search_interest END) AS search_interest14,
                    MAX(CASE WHEN extraction_window = 14
                             THEN increase_pct END)    AS increase_pct14

                FROM {GOOGLE_TOP_CLEANED}
                GROUP BY query),

            -- Choose the most relevant window for scoring.
            -- The 7-day window is preferred when available,
            -- otherwise the 14-day window is used.
            selected AS (
                SELECT
                    query,
                    b7,
                    b14,

                    -- Binary temporal fingerprint (b14b7)
                    CAST(b14 AS TEXT) || CAST(b7 AS TEXT) AS pattern,

                    CASE
                        WHEN b7 = 1 THEN 7
                        WHEN b14 = 1 THEN 14
                        ELSE NULL
                    END AS selected_window,

                    CASE
                        WHEN b7 = 1 THEN search_interest7
                        ELSE search_interest14
                    END AS selected_search_interest,

                    CASE
                        WHEN b7 = 1 THEN increase_pct7
                        ELSE increase_pct14
                    END AS selected_increase_pct
                FROM pivoted)

            -- Compute the final score using the same
            -- weighted combination used elsewhere in the project
            SELECT
                query,
                b7,
                b14,
                pattern,
                selected_window,
                selected_search_interest,
                selected_increase_pct,
                0.7 * selected_search_interest
              + 0.3 * selected_increase_pct AS final_score
            FROM selected
        """)

        # Ranking is not based solely on the score. Temporal patterns are applied first to prioritize recent
        # queries, and the composite score is used only to order queries within the same temporal group.

        cursor.execute(f"DROP TABLE IF EXISTS {GOOGLE_TOP_RANKED}")

        cursor.execute(f"""
            CREATE TABLE {GOOGLE_TOP_RANKED} AS
            WITH ranked AS (
                SELECT *,
                    CASE pattern
                        WHEN '11' THEN 1   -- present in both 14d and 7d
                        WHEN '01' THEN 2   -- present only in 7d (very recent)
                        WHEN '10' THEN 3   -- present only in 14d
                        WHEN '00' THEN 4   -- should not happen, lowest priority
                        ELSE 99
                    END AS pattern_rank
                FROM {GOOGLE_TOP_COMPUTED}),
            ordered AS (
                SELECT *,
                       ROW_NUMBER() OVER (
                           ORDER BY pattern_rank ASC, final_score DESC
                       ) AS rn,
                       COUNT(*) OVER () AS total_rows
                FROM ranked)
            -- Extract the top 20% of queries
            -- Always keep at least one query if the dataset is very small
            SELECT query
            FROM ordered
            WHERE rn <= CASE
                WHEN total_rows < 5 THEN 1
                ELSE CAST(0.3 * total_rows AS INTEGER)
            END
            ORDER BY rn
        """)

        conn.commit()

        print("Google Trends TOP analysis completed")

## Cleaning and Analysis on SQL pipeline

In [ ]:
# Executes the full data cleaning and analysis pipeline
    # for YouTube and Google Trends datasets.
def cleaning_and_analysis(topic: str,storage_path: str | Path,output_dir: str | Path):
    start_time = datetime.now(UTC)
    # YouTube data cleaning
    # Filters incomplete videos and channels and builds the clean YouTube statistics table.
    clean_youtube_data(topic=topic,storage_path=storage_path,output_dir=output_dir)

    # YouTube data analysis
    # Computes performance metrics and extracts high-potential videos and reels.
    analyze_youtube_data(topic=topic,storage_path=storage_path,output_dir=output_dir)

    # Google Trends RISING cleaning
    # Normalizes growth signals and flags breakout queries.
    rising_incomplete = clean_google_trends_rising(topic=topic,storage_path=storage_path,output_dir=output_dir)

    # Google Trends TOP cleaning
    # Normalizes short-term Google Trends TOP queries.
    top_incomplete = clean_google_trends_top(topic=topic,storage_path=storage_path,output_dir=output_dir)

    # Google Trends RISING analysis
    # Builds temporal patterns and ranks emerging queries.
    analyze_google_trends_rising(topic=topic,storage_path=storage_path,output_dir=output_dir)

    # Google Trends TOP analysis
    # Builds temporal patterns and ranks top queries.
    analyze_google_trends_top(topic=topic,storage_path=storage_path,output_dir=output_dir)

    end_time = datetime.now(UTC)
    print("\n            Cleaning and analizing completed\n")
    print(f"Cleaning and initial analysis concluded in {end_time - start_time}")

    return {"google_trends_rising_incomplete": rising_incomplete,
            "google_trends_top_incomplete": top_incomplete}


## Youtube EDA

In [3]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "Gill Sans","font.weight": "bold","axes.titleweight": "bold",
                     "axes.labelweight": "bold"})

# Normalize a keyword
# Converts spaces and hyphens to underscores and applies lowercase.
# This helper is used to generate compact and consistent identifiers
# for table names, file names, and internal references.
# Input : keyword (str)
# Output: str 
def normalize_keyword(keyword: str) -> str:
    return (keyword.lower().strip().replace(" ", "").replace("-", ""))


# Clean numeric data by removing commas and converting to float.
# Handles conversion errors by coercing invalid values to NaN,
# then filling missing values with zero to ensure numeric stability.
# Input : series (pd.Series)
# Output: pd.Series
def clean_numeric_series(series):
    return pd.to_numeric(series.astype(str).str.replace(',', ''), errors='coerce').fillna(0)

#ax.title.set_fontsize(title_size)
 #   ax.title.set_fontweight(title_weight)
  #  ax.title.set_fontname(font)
  #  ax.title.set_color(color)
  #  ax.title.set_ha("center")
def apply_axis_style(ax,title_size=25,label_size=18,tick_size=14,legend_size=14,title_weight="bold",
                     label_weight="bold",tick_weight="bold",font="Gill Sans",color="black",title_pad=8):
    # Title
    title = ax.get_title()
    ax.set_title(title,fontsize=title_size,fontweight=title_weight,fontname=font,color=color,pad=title_pad,
                 loc="center",)
    
    # Axis labels
    ax.xaxis.label.set_fontsize(label_size)
    ax.yaxis.label.set_fontsize(label_size)
    ax.xaxis.label.set_fontweight(label_weight)
    ax.yaxis.label.set_fontweight(label_weight)
    ax.xaxis.label.set_fontname(font)
    ax.yaxis.label.set_fontname(font)
    ax.xaxis.label.set_color(color)
    ax.yaxis.label.set_color(color)

    # Ticks
    ax.tick_params(axis="both", colors=color, labelsize=tick_size)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontname(font)
        lbl.set_fontweight(tick_weight)

    ax.grid(True, which="major", axis="y", linestyle="--", linewidth=0.8, alpha=0.35)
    ax.grid(True, which="minor", axis="y", linestyle=":", linewidth=0.5, alpha=0.2)
    ax.set_axisbelow(True)
    # Spines → NERI
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_color(color)
        spine.set_linewidth(1.2)

    # Legend → trasparente, no bordo, no titolo
    leg = ax.get_legend()
    if leg:
        leg.set_frame_on(False)
        leg.set_title(None)
        leg.get_frame().set_alpha(0.0)

        for text in leg.get_texts():
            text.set_fontsize(legend_size)
            text.set_fontweight(tick_weight)
            text.set_fontname(font)
            text.set_color(color)

# Unified textual EDA report generator
# Produces a structured, human-readable summary of the exploratory analysis
# for either the complete dataset or the filtered "interesting" subset.
#
# The report includes:
#   - dataset composition and content mix
#   - descriptive statistics for Videos and Reels
#   - statistical comparison via t-tests
#   - temporal performance aggregates (day-level)
#   - performance segmentation summaries
#   - top-performing content examples
#
# Input:
#   - df              : full analytical dataset
#   - df_videos       : subset containing only long-form videos
#   - df_reels        : subset containing only short-form reels
#   - stats_videos    : descriptive statistics table for videos
#   - stats_reels     : descriptive statistics table for reels
#   - ttest_df        : results of statistical significance tests
#   - output_dir      : directory where the report will be saved
#   - prefix          : topic identifier used for file naming
#   - is_interesting  : flag indicating whether the report refers to top content only
#
# Output:
#   - Path to the generated .txt report file
def generate_text_report_unified(df, df_videos, df_reels, stats_videos, stats_reels, 
                                 ttest_df, output_dir: str |Path, prefix: str, is_interesting: bool = False):
    
    # Select output filename based on analysis scope
    report_suffix = "interesting_eda_report" if is_interesting else "complete_eda_report"
    report_path = output_dir / f"{prefix}_{report_suffix}.txt"
    
    # Label used in the report header to clarify dataset scope
    analysis_type = "TOP PERFORMERS" if is_interesting else "COMPLETE DATASET"
    
    try:
        with open(report_path, 'w', encoding='utf-8') as f:
            # Report title and global context
            f.write(f"YouTube EDA Report - {prefix.upper()} ({analysis_type})\n")
            
            # Dataset composition and content type proportions
            f.write("    DATASET BREAKDOWN\n")
            total_records = len(df)
            f.write(f"Total Records: {total_records:,}\n")
            f.write(f"   Videos: {len(df_videos):,} ({(len(df_videos)/total_records)*100:.1f}%)\n")
            f.write(f"   Reels:  {len(df_reels):,} ({(len(df_reels)/total_records)*100:.1f}%)\n\n")
            
            # Descriptive statistics for each content type
            f.write("    DETAILED STATISTICS (Full Breakdown)\n")
            
            f.write("   VIDEOS STATISTICS:\n")
            f.write(stats_videos.round(2).to_string() + "\n\n")
            
            f.write("   REELS STATISTICS:\n")
            f.write(stats_reels.round(2).to_string() + "\n\n")

            # Hypothesis testing results comparing Videos and Reels
            f.write("    STATISTICAL SIGNIFICANCE (T-Test Results)\n")
            if not ttest_df.empty:
                for _, row in ttest_df.iterrows():
                    sig = "SIGNIFICANT" if row['p_value'] < 0.05 else "Not Significant"
                    direction = "Higher" if row['diff_pct'] > 0 else "Lower"
                    f.write(f"    {row['metric']:<15}: "
                            f"p={row['p_value']:.4f} {sig} | "
                            f"Reels Avg is {abs(row['diff_pct']):.1f}% {direction}\n")
            else:
                f.write("   Insufficient data to perform T-Tests.\n")
            f.write("\n")

            # Aggregated temporal performance indicators used for plotting
            f.write("    TEMPORAL ANALYSIS DATA (Graph Data)\n")
            
            if 'day_name' in df.columns:
                f.write("\n   Median Views by Day:\n")
                day_stats = df.groupby(['day_name', 'Type'], observed=False)['view_count'].median().unstack()
                f.write(day_stats.round(1).to_string() + "\n")
                
                f.write("\n   Best Day to Post (Highest Median Views):\n")
                if 'Videos' in day_stats.columns:
                    best_day_v = day_stats['Videos'].idxmax()
                    f.write(f"    Videos: {best_day_v} ({day_stats.loc[best_day_v, 'Videos']:,.0f} views)\n")
                if 'Reels' in day_stats.columns:
                    best_day_r = day_stats['Reels'].idxmax()
                    f.write(f"    Reels:  {best_day_r} ({day_stats.loc[best_day_r, 'Reels']:,.0f} views)\n")

            # Distribution of content types across performance tiers
            f.write("    SEGMENTATION DATA (Graph Data)\n")
            if 'segment' in df.columns:
                seg_counts = df.groupby(['segment', 'Type'], observed=False).size().unstack(fill_value=0)
                f.write("\n   Raw Counts by Segment:\n")
                f.write(seg_counts.to_string() + "\n")
                
                f.write("\n   Percentage Split:\n")
                seg_pct = seg_counts.div(seg_counts.sum(axis=1), axis=0) * 100
                f.write(seg_pct.round(1).to_string() + "\n")
            else:
                f.write("   Segmentation not available.\n")

        print(f"Full Detailed Text report saved: {report_path}")
    except Exception as e:
        print(f"Error saving text report: {e}")
        
    return report_path


# Plot: View count distribution
# Visualizes the distribution of video views using a logarithmic histogram.
# The plot highlights differences between Videos and Reels and
# mitigates heavy-tailed effects via log scaling.
# Input:
#   - ax (matplotlib.axes.Axes): Target axis for rendering
#   - df (DataFrame): Full cleaned dataset
#   - PALETTE (dict): Color mapping for content types
# Output:
#   - None (plot rendered on provided axis)
def plot_view_distribution(ax, df, PALETTE,is_interesting: bool = False):
            ax.set_facecolor("white")
            data = df[df["view_count"] > 0]
            if not data.empty:
                sns.histplot(data=data,x="view_count",hue="Type",palette=PALETTE,element="step",stat="count",
                             common_norm=False,kde=True,log_scale=True,ax=ax,)
            ax.set_title("View Distribution")
            subtitle="Interesting content" if is_interesting else "All content"
            ax.text(0.5, 1.0,subtitle,transform=ax.transAxes,ha="center",va="top",fontsize=10,fontfamily="Gill Sans",
                    fontweight="normal",color="black")
            ax.set_xlabel("View Count (log scale)")
            ax.set_ylabel("Frequency")
            apply_axis_style(ax)


# Plot: Videos vs Reels performance comparison
# Compares the distribution of views between Videos and Reels
# using a boxplot with logarithmic scaling.
# Outliers are suppressed to emphasize central tendencies.
# Input:
#   - ax (matplotlib.axes.Axes): Target axis for rendering
#   - df (DataFrame): Full cleaned dataset
#   - PALETTE (dict): Color mapping for content types
# Output:
#   - None (plot rendered on provided axis)
def plot_videos_vs_reels(ax, df, PALETTE, is_interesting: bool):
            ax.set_facecolor("white")
            data = df[df["view_count"] > 0]
            if not data.empty:sns.boxplot(data=data,x="Type",y="view_count",palette=PALETTE,showfliers=False,ax=ax,)
            ax.set_yscale("log")
            ax.set_title("Videos vs Reels")
            subtitle="Interesting content" if is_interesting else "All content"
            ax.text(0.5, 1.0,subtitle,transform=ax.transAxes,ha="center",va="top",fontsize=10,fontfamily="Gill Sans",
                    fontweight="normal",color="black")
            ax.set_xlabel("Content Type")
            ax.set_ylabel("Views (log scale)")
            apply_axis_style(ax)
    
# Plot: Weekly publishing trend
# Shows median view counts by day of the week for each content type.
# The plot supports identification of optimal publishing days.
# Input:
#   - ax (matplotlib.axes.Axes): Target axis for rendering
#   - df (DataFrame): Full cleaned dataset
#   - PALETTE (dict): Color mapping for content types
# Output:
#   - None (plot rendered on provided axis)
def plot_day_trend(ax, df, PALETTE, is_interesting: bool, white_bg: bool = False,):
    ax.set_facecolor("white" if white_bg else "#9FEEFE")

    d = (df.groupby(["day_name", "Type"], observed=False)["view_count"].median().reset_index())

    order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    if not d.empty:
        sns.lineplot(data=d,x="day_name",y="view_count",hue="Type",palette=PALETTE,marker="o",sort=False,
                     errorbar=None,ax=ax,)

    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=45)

    ax.set_title("Weekly Publishing Performance")
    subtitle="Interesting content" if is_interesting else "All content"
    ax.text(0.5, 1.0,subtitle,transform=ax.transAxes,ha="center",va="top",fontsize=10,fontfamily="Gill Sans",
            fontweight="normal",color="black")
    ax.set_xlabel("Day of week")
    ax.set_ylabel("Median views")

    ax.grid(True, alpha=0.15, linestyle="--")
    apply_axis_style(ax)

# Plot: Hourly engagement trend
# Displays median views by hour of publication for Videos and Reels.
# Useful for identifying peak engagement windows during the day.
# Input:
#   - ax (matplotlib.axes.Axes): Target axis for rendering
#   - df (DataFrame): Full cleaned dataset
#   - PALETTE (dict): Color mapping for content types
# Output:
#   - None (plot rendered on provided axis)
def plot_hour_trend(ax, df, PALETTE, is_interesting: bool, white_bg: bool = False):
    ax.set_facecolor("white" if white_bg else "#9FEEFE")

    h = (df.groupby(["hour", "Type"], observed=False)["view_count"].median().reset_index())

    if not h.empty:
        sns.lineplot(data=h,x="hour",y="view_count",hue="Type",palette=PALETTE,marker="o",errorbar=None,ax=ax,)

    ax.set_xticks(range(0, 24, 2))
    ax.set_title("Daily Publishing Performance")
    subtitle="Interesting content" if is_interesting else "All content"
    ax.text(0.5, 1.0,subtitle,transform=ax.transAxes,ha="center",va="top",fontsize=10,fontfamily="Gill Sans",
            fontweight="normal",color="black")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Median views")

    ax.grid(True, alpha=0.15, linestyle="--")
    apply_axis_style(ax)

# Plot: Content type mix by performance segment
# Visualizes the relative proportion of Videos and Reels
# across performance tiers derived from view count quantiles.
# Input:
#   - ax (matplotlib.axes.Axes): Target axis for rendering
#   - df (DataFrame): Full cleaned dataset
#   - PALETTE (dict): Color mapping for content types
# Output:
#   - None (plot rendered on provided axis)
def plot_segment_mix(ax, df, PALETTE, is_interesting: bool):
                c = (df.groupby(["segment", "Type"], observed=False).size().reset_index(name="count"))
                totals = c.groupby("segment", observed=False)["count"].transform("sum")
                c["pct"] = np.where(totals > 0, (c["count"] / totals) * 100, 0)
                if not c.empty:
                    sns.barplot(data=c,x="segment",y="pct",hue="Type",palette=PALETTE,ax=ax,)
                ax.set_title("Content Type Mix", fontweight="bold")
                subtitle="Interesting content" if is_interesting else "All content"
                ax.text(0.5, 1.0,subtitle,transform=ax.transAxes,ha="center",va="top",fontsize=10,
                        fontfamily="Gill Sans",fontweight="normal",color="black")
                ax.set_xlabel("Performance tier")
                ax.set_ylabel("Percentage (%)")
                ax.set_ylim(0, 100)
                apply_axis_style(ax)
    

# CONTENT SELECTION ANALYSIS
# Function: plot_best_vs_opportunity
# Builds a 2D content selection map by recomputing Best Content Score
# and Opportunity Score directly from the analytical dataset.
#
# Scores are recomputed on-the-fly using percentile-based normalization
# to ensure robustness and comparability across heterogeneous metrics.
# The function does NOT modify the input DataFrame and does NOT persist
# any derived values back to storage.
#
# The visualization highlights:
#   - the full universe of analyzed videos
#   - top-performing content ("Best")
#   - underexploited high-potential content ("Opportunity")
#   - videos satisfying both criteria simultaneously
#
# Input :
#   - ax (matplotlib.axes.Axes)   target axis for rendering the plot
#   - df (DataFrame)              dataset containing engagement, exposure
#                                 and view-related metrics
#
# Output:
#   - None → draws the scatter plot on the provided axis
def plot_best_vs_opportunity(ax, df, is_interesting: bool, white_bg: bool = False):
    
    # Final decision scores (single source of truth)
    best_score = df["best_content_score"]
    opp_score  = df["opportunity_score"]

    # Semantic flags from SQL
    is_best = df.get("is_best_score", 0).astype(bool)
    is_opp  = df.get("is_opportunity", 0).astype(bool)
    both    = is_best & is_opp

    # All content
    ax.scatter(best_score,opp_score,color="#8A8F98",s=15,zorder=1)
    ax.set_facecolor("white" if white_bg else "#9FEEFE")
    # Best content
    if (is_best & ~is_opp).any():
        ax.scatter(best_score[is_best & ~is_opp],opp_score[is_best & ~is_opp],color="#005F99",s=20,zorder=3,
                   label="Best content")
    # Opportunity content
    if (is_opp & ~is_best).any():
        ax.scatter(best_score[is_opp & ~is_best],opp_score[is_opp & ~is_best],color="#1B9E77",s=20,zorder=3,
                   label="Top opportunity")
    # Best & Opportunity
    if both.any():
        ax.scatter(best_score[both],opp_score[both],color="#FFB703",s=20,zorder=4,label="Best & Opportunity")

    ax.set_title("Content Selection Map")
    subtitle="Interesting content" if is_interesting else "All content"
    ax.text(0.5, 1.0,subtitle,transform=ax.transAxes,ha="center",va="top",fontsize=10,fontfamily="Gill Sans",
            fontweight="normal",color="black")
    ax.set_xlabel("Best Content Score")
    ax.set_ylabel("Opportunity Score")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.15, linestyle="--")
    apply_axis_style(ax)

NameError: name 'Path' is not defined

## Complete dataset analysis 

In [4]:
# COMPLETE EXPLORATORY DATA ANALYSIS
# Function: complete_eda_analysis
# Performs a comprehensive exploratory analysis on the FULL YouTube dataset stored in SQLite.
# The function loads the enriched statistics table, augments it with semantic flags
# (best content and opportunity selections), computes descriptive statistics,
# temporal performance aggregates, statistical tests, and generates analytical plots.
# Results are exported as CSV files, figures, and a unified textual report.
#
# Input :
#   - prefix (str)                    topic identifier used for table resolution
#   - sqlite_db_path (str | Path)     path to the SQLite database
#   - output_dir (str)                directory where plots and reports are saved
#   - export_dir (str | None)         directory for exported CSV aggregates
#   - generate_plots (bool)           whether to generate plots and dashboards
#   - is_interesting (bool)           semantic flag used at report level
#
# Output:
#   - dict → collection of analysis artifacts and intermediate results

def complete_eda_analysis(prefix: str,sqlite_db_path: str | Path,output_dir: str = "Analysis",
                          export_dir: str | None = None,generate_plots: bool = True,is_interesting: bool = False):

    # Normalize topic identifier to ensure consistent naming across the pipeline
    prefix = normalize_keyword(prefix)
    warnings.filterwarnings("ignore")

    # Initialize output directory for plots and reports
    OUTPUT_DIR = Path(output_dir)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Define shared color palette and plotting style
    PALETTE = {"Videos": "#005F99", "Reels": "#FF5757"}
    sns.set_style("whitegrid")

    # Log analysis start
    print(f"\nSTARTING EDA (COMPLETE DATASET): {prefix}")

    # Resolve and validate SQLite database path
    sqlite_db_path = Path(sqlite_db_path)
    if not sqlite_db_path.exists():
        raise FileNotFoundError(f"SQLite database not found: {sqlite_db_path}")

    # Enforce explicit export directory
    if export_dir is None:
        raise ValueError("export_dir must be provided explicitly")

    EXPORT_DIR = Path(export_dir)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    # Define SQL table names
    table_name = f"{prefix}_youtube_full_statistics"
    best_score_table = f"{prefix}_best_score_videos"
    opportunity_table = f"{prefix}_top_opportunity_videos"

    # Load full statistics table enriched with semantic flags via SQL joins
    print(f"Loading data from SQL table (with semantic flags): {table_name}")

    query = f"""
    SELECT
        f.*,
        CASE
            WHEN b.video_id IS NOT NULL THEN 1
            ELSE 0
        END AS is_best_score,
        CASE
            WHEN o.video_id IS NOT NULL THEN 1
            ELSE 0
        END AS is_opportunity
    FROM {table_name} AS f
    LEFT JOIN {best_score_table}  AS b ON f.video_id = b.video_id
    LEFT JOIN {opportunity_table} AS o ON f.video_id = o.video_id
    """

    with sqlite3.connect(sqlite_db_path, timeout=30) as conn:
        df = pd.read_sql_query(query, conn)

    # Abort analysis if dataset is empty
    if df.empty:
        raise ValueError(f"SQL query returned an empty dataset — cannot run EDA")

    print(f"Loaded {len(df):,} rows")

    # Standardize view count column naming
    df.rename(columns={"video_view_count": "view_count"}, inplace=True)

    # Clean and normalize numeric columns
    for col in ["view_count", "like_count", "comment_count", "duration_seconds"]:
        if col in df.columns:
            df[col] = clean_numeric_series(df[col])
        else:
            if col != "duration_seconds":
                df[col] = 0.0

    # Normalize reel flag and derive content type
    df["is_reel"] = df["is_reel"].astype(str).str.lower().isin(["true", "1", "yes", "t"])
    df["Type"] = df["is_reel"].map({True: "Reels", False: "Videos"})

    # Parse publication timestamp and extract temporal features
    if "published_at" in df.columns:
        df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
        df["day_name"] = df["published_at"].dt.day_name()
        df["hour"] = df["published_at"].dt.hour

    # Split dataset by content type
    df_videos = df[df["Type"] == "Videos"].copy()
    df_reels  = df[df["Type"] == "Reels"].copy()

    # Compute median views by day of week and content type
    day_table = (df.groupby(["day_name", "Type"], observed=False)["view_count"]
                   .median()
                   .reset_index())

    # Compute median views by hour of day and content type
    hour_table = (df.groupby(["hour", "Type"], observed=False)["view_count"]
                    .median()
                    .reset_index())

    # Export temporal aggregates
    day_table.to_csv(EXPORT_DIR / f"{prefix}_publishing_day_trend_complete.csv", index=False)
    hour_table.to_csv(EXPORT_DIR / f"{prefix}_publishing_hour_trend_complete.csv", index=False)

    # Metrics used for descriptive statistics and hypothesis testing
    stats_cols = ["view_count", "like_count", "comment_count"]

    # Descriptive statistics by content type
    stats_videos = df_videos[stats_cols].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]).T

    stats_reels = df_reels[stats_cols].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]).T

    # Welch t-tests between videos and reels
    ttest_data = []
    for m in stats_cols:
        v_data = df_videos[m].dropna()
        r_data = df_reels[m].dropna()
        if len(v_data) > 1 and len(r_data) > 1:
            _, p_val = stats.ttest_ind(v_data, r_data, equal_var=False)
            diff_pct = ((r_data.mean() - v_data.mean()) / max(v_data.mean(), 1)) * 100
            ttest_data.append({"metric": m,"p_value": p_val,"diff_pct": diff_pct})

    ttest_df = pd.DataFrame(ttest_data)

    # Segment videos into performance tiers based on view distribution
    try:
        df["segment"] = pd.qcut(df["view_count"], q=3,
                                labels=["Low", "Medium", "High"],
                                duplicates="drop")
    except:
        df["segment"] = "All"

    # Package analysis results
    results = {"raw_data": df,
               "stats_videos": stats_videos,
               "stats_reels": stats_reels,
               "ttest_df": ttest_df,}

    # Plot generation block
    if generate_plots:
        print("\nGENERATING PLOTS...")

        plot_tasks = [("view_distribution", lambda ax: plot_view_distribution(ax, df, PALETTE, False)),
                      ("videos_vs_reels",   lambda ax: plot_videos_vs_reels(ax, df, PALETTE,False)),]

        if "day_name" in df.columns:
            plot_tasks.append(("day_trend", lambda ax: plot_day_trend(ax, df, PALETTE, False, True)))

        if "hour" in df.columns:
            plot_tasks.append(("hour_trend", lambda ax: plot_hour_trend(ax, df, PALETTE,False, True)))

        if "segment" in df.columns:
            plot_tasks.append(("segment_mix", lambda ax: plot_segment_mix(ax, df, PALETTE, False)))

        if "best_content_score" in df.columns and "opportunity_score" in df.columns:
            plot_tasks.append(("best_vs_opportunity", lambda ax: plot_best_vs_opportunity(ax, df, False, True)))

        if "best_content_score" in df.columns and "opportunity_score" in df.columns:
            plot_tasks.append(("best_vs_opportunity_ppt", lambda ax: plot_best_vs_opportunity(ax, df,False, False)))

        # Generate and save individual plots
        saved_plots = []
        for i, (name, task) in enumerate(plot_tasks):
            fname = f"{i+1:02d}_complete_{name}.png"
            fig, ax = plt.subplots(figsize=(10, 6))
        
            if name.endswith("_ppt"):
                fig.patch.set_facecolor("#9FEEFE")   
            else:
                fig.patch.set_facecolor("white")
        
            try:
                task(ax)
                fig.savefig(OUTPUT_DIR / fname, dpi=150, bbox_inches="tight")
            except Exception as e:
                print(f"Error generating plot {fname}: {e}")
            plt.close(fig)


        # Generate combined dashboard
        white_tasks = [(name, task) for name, task in plot_tasks if not name.endswith("_ppt")]
        
        if white_tasks:
            rows = math.ceil(len(white_tasks) / 2)
            fig, axes = plt.subplots(rows, 2, figsize=(16, 6 * rows))
            fig.patch.set_facecolor("white")
            axes = axes.flatten()
        
            for i, (_, task) in enumerate(white_tasks):
                try:
                    task(axes[i])
                except Exception as e:
                    print(f"Dashboard plot error: {e}")
        
            for j in range(len(white_tasks), len(axes)):
                fig.delaxes(axes[j])
        
            plt.suptitle(f"Complete Content Performance Analytics - {prefix.upper()}",fontsize=24,fontweight="bold",
                         y=1.02)
        
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "Master_COMPLETE_Dashboard.png",dpi=200,bbox_inches="tight")
            plt.close(fig)

    # Generate unified textual report
    generate_text_report_unified(df, df_videos, df_reels,stats_videos, stats_reels, ttest_df,
                                 OUTPUT_DIR, prefix,is_interesting=is_interesting)

    return results


NameError: name 'Path' is not defined

## Interesting videos analysis

In [1]:
# Runs Exploratory Data Analysis on the INTERESTING (top-performing) dataset.
# This function mirrors the complete EDA logic but operates exclusively on
# the `{prefix}_interesting_statistics` table.
# The `is_interesting` flag is used only for report labeling.
# Input :
#   - prefix (str)                  topic identifier
#   - sqlite_db_path (str | Path)   path to the SQLite database
#   - output_dir (str | Path)       directory where plots and report are saved
#   - export_dir (str | Path | None) directory for CSV temporal aggregates
#   - generate_plots (bool)         whether to generate plots and dashboard
#   - is_interesting (bool)         report-level flag (semantic only)
# Output:
#   - dict                          collection of analysis artifacts

def interesting_eda_analysis(prefix: str,sqlite_db_path: str | Path,output_dir: str = "Analysis",
                             export_dir: str | None = None,generate_plots: bool = True,is_interesting: bool = True):
   
        # Normalize topic identifier to ensure consistency across tables and files
        prefix = normalize_keyword(prefix)
        # Suppress non-critical warnings for cleaner logs
        warnings.filterwarnings("ignore")
    
        # Initialize output directory for plots and reports
        OUTPUT_DIR = Path(output_dir)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
        # Define shared color palette for all visualizations
        PALETTE = {"Videos": "#005F99", "Reels": "#FF5757"}
        sns.set_style("whitegrid")
    
        # Log analysis start
        print(f"\nSTARTING EDA (INTERESTING DATASET): {prefix}")
    
        # Resolve and validate SQLite database path
        sqlite_db_path = Path(sqlite_db_path)
        if not sqlite_db_path.exists():
            raise FileNotFoundError(f"SQLite database not found: {sqlite_db_path}")
    
        # Select the INTERESTING statistics table
        table_name = f"{prefix}_interesting_statistics"
        print(f"Loading data from SQL table: {table_name}")
    
        # Load data from SQLite into a DataFrame
        with sqlite3.connect(sqlite_db_path, timeout=30) as conn:
            df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    
        # Abort analysis if table is empty
        if df.empty:
            raise ValueError(f"SQL table '{table_name}' is empty — cannot run EDA")
    
        # Log number of loaded rows
        print(f"Loaded {len(df):,} rows")
    
        # Standardize view count column naming
        df.rename(columns={"video_view_count": "view_count"}, inplace=True)
    
        # Clean and normalize numeric columns used in the analysis
        for col in ["view_count", "like_count", "comment_count", "duration_seconds"]:
            if col in df.columns:
                df[col] = clean_numeric_series(df[col])
            else:
                # Fill missing numeric metrics with zero (except duration)
                if col != "duration_seconds":
                    df[col] = 0.0
    
        # Normalize reel flag and derive human-readable content type
        df["is_reel"] = df["is_reel"].astype(str).str.lower().isin(["true", "1", "yes", "t"])
        df["Type"] = df["is_reel"].map({True: "Reels", False: "Videos"})
    
        # Parse publication timestamp and extract temporal features
        if "published_at" in df.columns:
            df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
            df["day_name"] = df["published_at"].dt.day_name()
            df["hour"] = df["published_at"].dt.hour
    
        # Split dataset by content type for comparative statistics
        df_videos = df[df["Type"] == "Videos"].copy()
        df_reels = df[df["Type"] == "Reels"].copy()
    
        # Validate export directory for aggregated temporal outputs
        if export_dir is None:
            raise ValueError("export_dir must be provided explicitly")
    
        EXPORT_DIR = Path(export_dir)
        EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    
        # Compute median views by day of week and content type
        day_table = (df.groupby(["day_name", "Type"], observed=False)["view_count"].median().reset_index())
    
        # Compute median views by hour of day and content type
        hour_table = (df.groupby(["hour", "Type"], observed=False)["view_count"].median().reset_index())
    
        # Export temporal aggregates to CSV
        day_table.to_csv(EXPORT_DIR / f"{prefix}_publishing_day_trend_interesting.csv",index=False)
        hour_table.to_csv(EXPORT_DIR / f"{prefix}_publishing_hour_trend_interesting.csv",index=False)

        # Loadin tables on SQLite
        with sqlite3.connect(sqlite_db_path, timeout=30) as conn:
            day_table.to_sql(f"{prefix}_interesting_publishing_day",conn,if_exists="replace",index=False)

            hour_table.to_sql(f"{prefix}_interesting_publishing_hour",conn,if_exists="replace",index=False)

        print("Loaded interesting publishing day/hour tables to SQLite")

    
        # Define metrics used for descriptive statistics and tests
        stats_cols = ["view_count", "like_count", "comment_count"]
    
        # Compute descriptive statistics for videos
        stats_videos = df_videos[stats_cols].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]).T
    
        # Compute descriptive statistics for reels
        stats_reels = df_reels[stats_cols].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]).T
    
        # Perform Welch t-tests between videos and reels
        ttest_data = []
        for m in stats_cols:
            v_data = df_videos[m].dropna()
            r_data = df_reels[m].dropna()
            if len(v_data) > 1 and len(r_data) > 1:
                t_stat, p_val = stats.ttest_ind(v_data, r_data, equal_var=False)
                diff_pct = ((r_data.mean() - v_data.mean()) / max(v_data.mean(), 1)) * 100
                ttest_data.append({"metric": m, "p_value": p_val, "diff_pct": diff_pct})
    
        ttest_df = pd.DataFrame(ttest_data)
    
        # Segment content into performance tiers based on view distribution
        try:
            df["segment"] = pd.qcut(df["view_count"],q=3,labels=["Low", "Medium", "High"],duplicates="drop",)
        except:
            df["segment"] = "All"
    
        # Package analysis results
        results = {"raw_data": df,
                   "stats_videos": stats_videos,
                   "stats_reels": stats_reels,
                   "ttest_df": ttest_df,}
    
        if generate_plots:
            print("\nGENERATING PLOTS...")
    
            # Initialize ordered list of plotting tasks
            plot_tasks = [("view_distribution", lambda ax: plot_view_distribution(ax, df, PALETTE,True)),
                          ("videos_vs_reels",   lambda ax: plot_videos_vs_reels(ax, df, PALETTE,True)),]

    
            # Add temporal plots if features are available
            if "day_name" in df.columns:
                plot_tasks.append(("day_trend", lambda ax: plot_day_trend(ax, df, PALETTE, True, True)))

            if "hour" in df.columns:
                plot_tasks.append(("hour_trend", lambda ax: plot_hour_trend(ax, df, PALETTE, True, True)))
    
            # Add segmentation plot if segmentation was computed
            if "segment" in df.columns:
                plot_tasks.append(("segment_mix", lambda ax: plot_segment_mix(ax, df, PALETTE, True)))

            if "day_name" in df.columns:
                plot_tasks.append(("day_trend_ppt", lambda ax: plot_day_trend(ax, df, PALETTE,True, False)))

            if "hour" in df.columns:
                plot_tasks.append(("hour_trend_ppt", lambda ax: plot_hour_trend(ax, df, PALETTE,True,False)))
            
            
            # Generate and save individual plots
            saved_plots = []
            for i, (name, task) in enumerate(plot_tasks):
                fname = f"{i+1:02d}_interesting_{name}.jpg"
                fig, ax = plt.subplots(figsize=(10, 6))
                if name.endswith("_ppt"):
                    fig.patch.set_facecolor("#9FEEFE")   
                else:
                    fig.patch.set_facecolor("white")
                try:
                    task(ax)
                    fig.savefig(OUTPUT_DIR / fname, dpi=150, bbox_inches="tight")
                    saved_plots.append(fname)
                except Exception as e:
                    print(f"Error generating plot {fname}: {e}")
                plt.close(fig)
    
            # Build combined dashboard if plots were generated
            white_tasks = [(name, task) for name, task in plot_tasks if not name.endswith("_ppt")]
            
            if white_tasks:
                rows = math.ceil(len(white_tasks) / 2)
                fig, axes = plt.subplots(rows, 2, figsize=(16, 6 * rows))
                fig.patch.set_facecolor("white")
                axes = axes.flatten()
            
                for i, (_, task) in enumerate(white_tasks):
                    try:
                        task(axes[i])
                    except Exception as e:
                        print(f"Dashboard plot error: {e}")
            
                for j in range(len(white_tasks), len(axes)):
                    fig.delaxes(axes[j])
            
                plt.suptitle(f"Top Content Performance Analytics - {prefix.upper()}",fontsize=24,fontweight="bold",
                             y=1.02)
                plt.tight_layout()
                plt.savefig(OUTPUT_DIR / "Master_TOP_Dashboard.png",dpi=200,bbox_inches="tight",)
                plt.close(fig)
    
        # Generate unified textual report for the interesting dataset
        generate_text_report_unified(df,df_videos,df_reels,stats_videos,stats_reels,ttest_df,
                                     OUTPUT_DIR,prefix,is_interesting=is_interesting,)
        if not ttest_df.empty and len(ttest_df.columns) > 0:
            ttest_df = ttest_df.copy()
            
            ttest_df.columns = [str(c).strip().replace(" ", "_").replace("%", "pct")
                                if c not in (None, "", " ") else f"col_{i}" for i, c in enumerate(ttest_df.columns)]
    
            with sqlite3.connect(sqlite_db_path, timeout=30) as conn:
                ttest_df.to_sql(f"{prefix}_significance_tests",conn,if_exists="replace",index=False)
        
    
        # Return analysis artifacts
        return results


NameError: name 'Path' is not defined

## EDA pipeline

In [ ]:
# EDA ORCHESTRATION
# Function: run_EDA
# Orchestrates the execution of exploratory data analysis on both:
#   1) the COMPLETE YouTube dataset
#   2) the INTERESTING subset (top-performing / high-opportunity content)
#
# The function acts as a thin coordination layer that:
#   - validates paths and directories
#   - enforces consistent topic normalization
#   - measures execution time for each EDA stage
#   - delegates analytical work to specialized EDA functions
#
# Input :
#   - prefix (str)                    topic identifier
#   - sqlite_db_path (str | Path)     path to the SQLite database
#   - complete_output_dir (str|Path)  directory for complete-dataset outputs
#   - interesting_output_dir (str|Path) directory for interesting-dataset outputs
#   - export_dir (str | Path)         directory for shared CSV exports
#   - generate_plots (bool)           whether plots and dashboards are generated
#
# Output:
#   - None (side effects: files, plots, reports written to disk)

def run_EDA(prefix: str,sqlite_db_path: str | Path,complete_output_dir: str | Path,
            interesting_output_dir: str | Path,export_dir: str | Path,generate_plots: bool = True,):

    # Normalize topic identifier to ensure consistency across the pipeline
    prefix = normalize_keyword(prefix)

    # Resolve and validate SQLite database path
    sqlite_db_path = Path(sqlite_db_path)
    if not sqlite_db_path.exists():
        raise FileNotFoundError(f"SQLite database not found: {sqlite_db_path}")

    # Resolve output directories
    complete_output_dir = Path(complete_output_dir)
    interesting_output_dir = Path(interesting_output_dir)
    export_dir = Path(export_dir)

    # Ensure all required directories exist
    complete_output_dir.mkdir(parents=True, exist_ok=True)
    interesting_output_dir.mkdir(parents=True, exist_ok=True)
    export_dir.mkdir(parents=True, exist_ok=True)

    print("\nEDA on complete dataset")
    start_time = datetime.now(UTC)

    complete_eda_analysis(prefix=prefix,sqlite_db_path=sqlite_db_path,output_dir=complete_output_dir,
                          export_dir=export_dir,generate_plots=generate_plots,is_interesting=False,)

    end_time = datetime.now(UTC)
    print(f"Complete dataset EDA concluded in {end_time - start_time}")

    print("\nEDA on TOP performer")
    start_time = datetime.now(UTC)

    interesting_eda_analysis(prefix=prefix,sqlite_db_path=sqlite_db_path,output_dir=interesting_output_dir,
                             export_dir=export_dir,generate_plots=generate_plots,is_interesting=True,)

    end_time = datetime.now(UTC)
    print(f"Interesting dataset EDA concluded in {end_time - start_time}")

    print("\nExploratory data analysis concluded\n")
